# ML-03 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amah67/mlintern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)



## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis – One entry represents one unique piece of published content (content_id) per particular client (client_id).

Time Window – The features are calculated based on the initial performance metrics within the observation period, whereas the target is computed based on the change in performance observed during the next period.

In [2]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Verify grain: duplicate count must be 0
duplicate_grain = df.duplicated(subset=["content_id"]).sum()
print(f"Total Rows: {len(df):,}")
print(f"Unique content_ids: {df['content_id'].nunique():,}")
print(f"Duplicate content_ids: {duplicate_grain} (Grain holds if 0)")

# Check date/window columns if present
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]
if date_cols:
  for c in date_cols:
    print(f"Window for {c}: {df[c].min()} to {df[c].max()}")
else:
  print(
      "Static performance snapshot dataset: time window defined across feature"
      " vs outcome aggregation periods."
  )

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.
Total Rows: 30,000
Unique content_ids: 30,000
Duplicate content_ids: 0 (Grain holds if 0)
Window for days_since_last_update: 1 to 373


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context (IDs/Grouping) - content_id, client_id (IDs used only for splitting, grouping, and deduplication purposes; never used for feature engineering or modeling).

Features (Predictive Features) - search_volume, competition, competition_level, cpc, content_type, main_intent, word_count, char_count, char_count_tier, impression_tier, position_tier, ai_traffic_pct.

Target / Proxy Target (Observation) - trend_pct, trend_direction (observed change in search traffic visibility).

Excluded - scroll_rate, engagement_rate (not used in the model due to target leakage).

In [3]:
# Field buckets
context_cols = ["content_id", "client_id"]
feature_cols = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "ai_traffic_pct",
]
label_cols = ["trend_pct", "trend_direction"]
excluded_cols = [
    c
    for c in ["scroll_rate", "engagement_rate", "ctr", "avg_position"]
    if c in df.columns and c not in feature_cols + label_cols + context_cols
]

print("Field Classification Summary:")
print(f"Context Fields: {len(context_cols)} columns -> {context_cols}")
print(f"Feature Fields: {len(feature_cols)} columns -> {feature_cols}")
print(f"Label Fields:   {len(label_cols)} columns -> {label_cols}")
print(f"Excluded:       {len(excluded_cols)} columns -> {excluded_cols}")

Field Classification Summary:
Context Fields: 2 columns -> ['content_id', 'client_id']
Feature Fields: 12 columns -> ['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'char_count_tier', 'impression_tier', 'position_tier', 'ai_traffic_pct']
Label Fields:   2 columns -> ['trend_pct', 'trend_direction']
Excluded:       4 columns -> ['scroll_rate', 'engagement_rate', 'ctr', 'avg_position']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Ensured content_id is unique for all rows (no duplicates), counted total number of rows, calculated missing value percent for all features and target columns, and ensured there is adequate observation volume for target classes.

In [4]:
# 1. Grain Check
assert df.duplicated(subset=["content_id"]).sum() == 0, "Grain check failed!"
print("✔ Grain verified: 1 row = 1 unique content_id")

# 2. Total Counts
print(f"\nTotal Dataset Rows: {len(df):,}")
print(f"Total Unique Clients: {df['client_id'].nunique():,}")

# 3. Missingness Check
print("\nMissing Value Rate per Field:")
missingness = (df.isnull().sum() / len(df) * 100).round(2)
print(missingness[missingness > 0].to_string() if missingness.max() > 0 else "0% missing values across all columns.")

# 4. Target Distribution
if "trend_direction" in df.columns:
    print("\nObserved Target Distribution (trend_direction):")
    print(df["trend_direction"].value_counts(normalize=True).round(4) * 100)

✔ Grain verified: 1 row = 1 unique content_id

Total Dataset Rows: 30,000
Total Unique Clients: 32

Missing Value Rate per Field:
search_volume         8.23
competition           8.23
competition_level     8.70
cpc                   8.23
main_intent           7.91
word_count           25.66
char_count           25.66
provider_used        71.46
model_used           19.11
word_count_tier      25.66
char_count_tier      25.66
scroll_rate           0.42
trend_pct            11.29

Observed Target Distribution (trend_direction):
trend_direction
down      54.21
stable    19.87
up        14.63
new        7.45
flat       3.84
Name: proportion, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced History of the Clients - Not all clients have equal amounts of recorded search signals, and for small clients, there might be noisy search signals.

No Causal Information - The information in the database only gives the correlation between search signals and the rank of pages and does not help understand why the Google algorithm has changed its ranks.

External Algorithm Changes - Other variables affecting the ranking process are not observable from the database at the individual page level.